> **Prerequisites:**  
> - Notebook 01 must be completed (PDFs loaded and processed)
> - Notebook 02 must be completed (structured data imported)
> - Your `.env` file must contain valid Neo4j credentials

## Imports and Environment Setup

This cell imports all necessary libraries for connecting to Neo4j and displaying results.

In [1]:
%pip install -r ../../../requirements.txt


[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from neo4j import GraphDatabase
from dotenv import load_dotenv
import os
import pandas as pd

## Load Environment Variables and Connect to Neo4j

This cell loads environment variables required for connecting to Neo4j. It uses `python-dotenv` to securely access credentials from a `.env` file, and then establishes a connection to the Neo4j database using the provided URI, username, and password.

- `NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD`: Used for Neo4j database authentication.

In [3]:
load_dotenv()
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USER = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

In [4]:
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
print(f"Connected to Neo4j at {NEO4J_URI}")

Connected to Neo4j at neo4j+s://339bd49f.databases.neo4j.io


## Helper Function to Run Queries

This function executes a Cypher query and returns results as a pandas DataFrame for easy viewing.

In [5]:
def run_query(query):
    with driver.session() as session:
        result = session.run(query)
        return pd.DataFrame([dict(record) for record in result])

## Node Count Summary

Display a count of all node types in the database.

In [6]:
query = """
MATCH (n)
UNWIND labels(n) as label
RETURN label as NodeType, count(DISTINCT n) as Count
ORDER BY Count DESC
"""
run_query(query)

,NodeType,Count
0,__KGBuilder__,1864
1,__Entity__,1466
2,RiskFactor,751
3,Chunk,390
4,FinancialMetric,331
5,Product,311
6,Executive,25
7,StockType,16
8,TimePeriod,15
9,AssetManager,15


## Relationship Count Summary

Display a count of all relationship types in the database.

In [7]:
query = """
MATCH ()-[r]-()
RETURN type(r) as RelationshipType, count(*) as Count
ORDER BY Count DESC
"""
run_query(query)

,RelationshipType,Count
0,FROM_CHUNK,4330
1,FACES_RISK,1528
2,FROM_DOCUMENT,780
3,NEXT_CHUNK,764
4,HAS_METRIC,674
5,MENTIONS,626
6,OWNS,236
7,ISSUED_STOCK,34
8,FILED,20


## Database Schema

Display the complete database schema as JSON to see all node labels, relationship types, and properties.

In [8]:
query = """
CALL apoc.meta.schema()
YIELD value
RETURN value
"""
run_query(query)

,value
0,"{'Company': {'count': 11, 'relationships': {'F..."


### All Node Labels in Database

List all unique node labels found in the database.

In [9]:
query = """
CALL db.labels()
YIELD label
RETURN label
ORDER BY label
"""
run_query(query)

,label
0,AssetManager
1,Chunk
2,Company
3,Document
4,Executive
5,FinancialMetric
6,Product
7,RiskFactor
8,StockType
9,TimePeriod


### All Relationship Types in Database

List all unique relationship types found in the database.

In [10]:
query = """
CALL db.relationshipTypes()
YIELD relationshipType
RETURN relationshipType
ORDER BY relationshipType
"""
run_query(query)

,relationshipType
0,FACES_RISK
1,FILED
2,FROM_CHUNK
3,FROM_DOCUMENT
4,HAS_METRIC
5,ISSUED_STOCK
6,MENTIONS
7,NEXT_CHUNK
8,OWNS


### Sample of All Nodes with Properties

Get a sample of each node type to see what properties exist.

In [11]:
query = """
MATCH (n)
WITH labels(n)[0] as Label, collect(n)[0] as SampleNode
RETURN Label, keys(SampleNode) as Properties
ORDER BY Label
"""
run_query(query)

,Label,Properties
0,AssetManager,"[managerName, managerCik]"
1,Company,"[name, ticker]"
2,Document,[path]
3,__KGBuilder__,"[path, createdAt, document_type]"


---
## Verify Notebook 01 Data (PDF Loader)

### Documents

First 10 documents loaded from PDF files.

In [12]:
query = """
MATCH (d:Document)
RETURN d.path as DocumentPath
LIMIT 10
"""
run_query(query)

,DocumentPath
0,data/form10k-sample/0000950170-23-035122.pdf
1,data/form10k-sample/0001004980-23-000029.pdf
2,data/form10k-sample/0001652044-16-000012.pdf
3,data/form10k-sample/0001045810-23-000017.pdf
4,data/form10k-sample/0001018724-23-000004.pdf
5,data/form10k-sample/000063908-23-000012.pdf
6,data/form10k-sample/0001096906-23-001489.pdf
7,data/form10k-sample/000050863-23-000006.pdf
8,data/form10k-sample/00005272-23-000007.pdf
9,data/form10k-sample/0000320193-23-000106.pdf


### Chunks

First 10 text chunks extracted from documents, showing the text content and which document they came from.

In [13]:
query = """
MATCH (c:Chunk)-[:FROM_DOCUMENT]->(d:Document)
RETURN c.text as ChunkText, d.path as SourceDocument
LIMIT 10
"""
run_query(query)

,ChunkText,SourceDocument
0,10-K Filing Data\nSection: Item1\n>ITEM 1. B\n...,data/form10k-sample/0000950170-23-035122.pdf
1,"that brings\ntogether communications, knowledg...",data/form10k-sample/0000950170-23-035122.pdf
2,"telecommunications, joined Microsoft in 2022. ...",data/form10k-sample/0000950170-23-035122.pdf
3,intelligent solutions for our customers that e...,data/form10k-sample/0000950170-23-035122.pdf
4,Equity Initiative focuses on three multi-year...,data/form10k-sample/0000950170-23-035122.pdf
5,"igher education institutions,\nproviding train...",data/form10k-sample/0000950170-23-035122.pdf
6,opportunity).\n-\nDisability representation.\n...,data/form10k-sample/0000950170-23-035122.pdf
7,"decision making,\nbalancing the needs of busin...",data/form10k-sample/0000950170-23-035122.pdf
8,"users, add value to our core\nproduct set, and...",data/form10k-sample/0000950170-23-035122.pdf
9,"security solutions across email security,\ninf...",data/form10k-sample/0000950170-23-035122.pdf


### Companies (from PDFs)

First 10 companies extracted from PDF documents.

In [14]:
query = """
MATCH (c:Company)
WHERE c.name IS NOT NULL
RETURN c.name as CompanyName
LIMIT 10
"""
run_query(query)

,CompanyName
0,ALPHABET INC
1,AMAZON
2,AMERICAN INTL GROUP
3,APPLE INC
4,INTEL CORP
5,MCDONALDS CORP
6,MICROSOFT CORP
7,NVIDIA CORPORATION
8,PAYPAL
9,PAYPAL HLDGS INC


### Executives

First 10 executives extracted from documents.

In [15]:
query = """
MATCH (e:Executive)
RETURN e.name as ExecutiveName
LIMIT 10
"""
run_query(query)

,ExecutiveName
0,Satya Nadella
1,Judson B. Althoff
2,Christopher C. Capossela
3,Kathleen T. Hogan
4,Amy E. Hood
5,Bradford L. Smith
6,Christopher D. Young
7,Mr. Capossela
8,Ms. Hogan
9,Ms. Hood


### Products

First 10 products mentioned in documents.

In [16]:
query = """
MATCH (p:Product)
RETURN p.name as ProductName
LIMIT 10
"""
run_query(query)

,ProductName
0,Microsoft Cloud
1,Microsoft Teams
2,Outlook
3,Bing
4,Xbox
5,Office 365
6,Dynamics 365
7,LinkedIn
8,Microsoft 365
9,Windows


### Financial Metrics

First 10 financial metrics extracted from documents.

In [17]:
query = """
MATCH (fm:FinancialMetric)
RETURN fm.name as MetricName
LIMIT 10
"""
run_query(query)

,MetricName
0,Total Rewards and Pay Equity
1,Net income
2,Earnings per share
3,debt reduction by $2 billion by 2026
4,Net cash from operations
5,Net cash used in financing
6,Net cash used in investing
7,Effect of foreign exchange rates on cash and c...
8,Net change in cash and cash equivalents
9,"Cash and cash equivalents, end of period"


### Risk Factors

First 10 risk factors mentioned in documents.

In [18]:
query = """
MATCH (rf:RiskFactor)
RETURN rf.name as RiskFactor
LIMIT 10
"""
run_query(query)

,RiskFactor
0,Disability representation
1,Pay equity
2,Intellectual Property Risks
3,intense competition across all markets
4,low barriers to entry and rapid evolution in t...
5,competition from vertically-integrated models
6,significant competition for platform-based eco...
7,competition from new devices and form factors ...
8,Business model competition
9,Technological change


### Company Relationships

First 10 relationships showing companies connected to their financial metrics.

In [19]:
query = """
MATCH (c:Company)-[:HAS_METRIC]->(fm:FinancialMetric)
RETURN c.name as Company, fm.name as Metric
LIMIT 10
"""
run_query(query)

,Company,Metric
0,MICROSOFT CORP,Total Rewards and Pay Equity
1,MICROSOFT CORP,Net income
2,MICROSOFT CORP,Earnings per share
3,MICROSOFT CORP,Gross derivative assets and liabilities
4,MICROSOFT CORP,Derivative assets
5,MICROSOFT CORP,Derivative liabilities
6,MICROSOFT CORP,Gains (losses) on derivative instruments
7,MICROSOFT CORP,Gross amounts of derivatives
8,MICROSOFT CORP,Inventories
9,MICROSOFT CORP,Effective Tax Rate


### Company Risk Relationships

First 10 relationships showing companies and their risk factors.

In [20]:
query = """
MATCH (c:Company)-[:FACES_RISK]->(rf:RiskFactor)
RETURN c.name as Company, rf.name as RiskFactor
LIMIT 10
"""
run_query(query)

,Company,RiskFactor
0,MICROSOFT CORP,Disability representation
1,MICROSOFT CORP,Pay equity
2,MICROSOFT CORP,Intellectual Property Risks
3,MICROSOFT CORP,intense competition across all markets
4,MICROSOFT CORP,low barriers to entry and rapid evolution in t...
5,MICROSOFT CORP,competition from vertically-integrated models
6,MICROSOFT CORP,significant competition for platform-based eco...
7,MICROSOFT CORP,competition from new devices and form factors ...
8,MICROSOFT CORP,Business model competition
9,MICROSOFT CORP,Technological change


---
## Verify Notebook 02 Data (Structured Data)

### Companies with Tickers

First 10 companies from the structured data import, showing company names and ticker symbols.

In [21]:
query = """
MATCH (c:Company)
WHERE c.ticker IS NOT NULL
RETURN c.name as Company, c.ticker as Ticker
LIMIT 10
"""
run_query(query)

,Company,Ticker
0,MICROSOFT CORP,MSFT
1,AMAZON,AMZN
2,APPLE INC,AAPL
3,MCDONALDS CORP,MCD
4,PG&E CORP,PCG
5,AMERICAN INTL GROUP,AIG
6,NVIDIA CORPORATION,NVDA
7,INTEL CORP,INTC
8,PAYPAL HLDGS INC,PYPL
9,ALPHABET INC,GOOG


### Asset Managers

First 10 asset managers from the structured data.

In [22]:
query = """
MATCH (am:AssetManager)
RETURN am.managerName as AssetManager
LIMIT 10
"""
run_query(query)

,AssetManager
0,ALLIANCEBERNSTEIN L.P.
1,AMERIPRISE FINANCIAL INC
2,AMUNDI
3,BANK OF AMERICA CORP /DE/
4,Bank of New York Mellon Corp
5,Berkshire Hathaway Inc
6,BlackRock Inc.
7,Capital World Investors
8,FMR LLC
9,"GEODE CAPITAL MANAGEMENT, LLC"


### Asset Manager Ownership

First 10 OWNS relationships showing which asset managers own shares in which companies.

In [23]:
query = """
MATCH (am:AssetManager)-[o:OWNS]->(c:Company)
RETURN am.managerName as AssetManager, c.name as Company, o.shares as Shares
LIMIT 10
"""
run_query(query)

,AssetManager,Company,Shares
0,ALLIANCEBERNSTEIN L.P.,MICROSOFT CORP,46541943
1,AMERIPRISE FINANCIAL INC,MICROSOFT CORP,34839303
2,AMUNDI,MICROSOFT CORP,32644371
3,BANK OF AMERICA CORP /DE/,MICROSOFT CORP,70768993
4,Bank of New York Mellon Corp,MICROSOFT CORP,70582604
5,BlackRock Inc.,MICROSOFT CORP,533634606
6,Capital World Investors,MICROSOFT CORP,88620338
7,FMR LLC,MICROSOFT CORP,215874152
8,"GEODE CAPITAL MANAGEMENT, LLC",MICROSOFT CORP,151218886
9,MORGAN STANLEY,MICROSOFT CORP,123130465


### Company Filings

First 10 FILED relationships connecting companies to their documents (linking structured and unstructured data).

In [24]:
query = """
MATCH (c:Company)-[:FILED]->(d:Document)
RETURN c.name as Company, c.ticker as Ticker, d.path as DocumentPath
LIMIT 10
"""
run_query(query)

,Company,Ticker,DocumentPath
0,MICROSOFT CORP,MSFT,data/form10k-sample/0000950170-23-035122.pdf
1,PG&E CORP,PCG,data/form10k-sample/0001004980-23-000029.pdf
2,NVIDIA CORPORATION,NVDA,data/form10k-sample/0001045810-23-000017.pdf
3,AMAZON,AMZN,data/form10k-sample/0001018724-23-000004.pdf
4,MCDONALDS CORP,MCD,data/form10k-sample/000063908-23-000012.pdf
5,APPLE INC,AAPL,data/form10k-sample/0001096906-23-001489.pdf
6,INTEL CORP,INTC,data/form10k-sample/000050863-23-000006.pdf
7,AMERICAN INTL GROUP,AIG,data/form10k-sample/00005272-23-000007.pdf
8,APPLE INC,AAPL,data/form10k-sample/0000320193-23-000106.pdf
9,PAYPAL,None,data/form10k-sample/0001633917-23-000033.pdf


---
## Verification Complete

✅ If you see data in all the queries above, your knowledge graph has been successfully populated with both unstructured (PDF) and structured (CSV) data.

### Next Steps:
- Proceed to the retrieval and agent notebooks to query this data
- Explore the graph using Neo4j Browser or Bloom
- Run custom queries to analyze your specific use case

## Close Connection

Clean up by closing the database connection.

In [25]:
driver.close()
print("Connection closed")

Connection closed
